In [1]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install -q ultralytics requests tqdm

import os, json, requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

print("Setup complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.8 MB/s eta 0:00:00
Setup complete.


In [2]:
# ============================================================
# CELL 2: Upload the .jsonl manifest file (the one you pasted above)
# ============================================================
from google.colab import files

print("Upload your kitchen-hygiene-gear-4 .jsonl manifest file:")
uploaded = files.upload()
JSONL_PATH = list(uploaded.keys())[0]
print(f"Using manifest: {JSONL_PATH}")

Upload your kitchen-hygiene-gear-4 .jsonl manifest file:


Saving kitchen-hygiene-gear-4.ndjson to kitchen-hygiene-gear-4.ndjson
Using manifest: kitchen-hygiene-gear-4.ndjson


In [3]:
# ============================================================
# CELL 3: Parse JSONL -> separate dataset metadata + image records
# ============================================================
DATASET_ROOT = Path("/content/kitchen_hygiene_gear")
IMAGES_DIR = DATASET_ROOT / "images"
LABELS_DIR = DATASET_ROOT / "labels"

for split in ["train", "val", "test"]:
    (IMAGES_DIR / split).mkdir(parents=True, exist_ok=True)
    (LABELS_DIR / split).mkdir(parents=True, exist_ok=True)

dataset_meta = None
image_records = []

with open(JSONL_PATH, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        if obj["type"] == "dataset":
            dataset_meta = obj
        elif obj["type"] == "image":
            image_records.append(obj)

CLASS_NAMES = dataset_meta["class_names"]  # {"0":"glove", ...}
NUM_CLASSES = len(CLASS_NAMES)
CLASS_LIST = [CLASS_NAMES[str(i)] for i in range(NUM_CLASSES)]

print(f"Dataset: {dataset_meta['name']}")
print(f"Classes ({NUM_CLASSES}): {CLASS_LIST}")
print(f"Total image records: {len(image_records)}")

# Map Ultralytics 'split' field -> our folder names (test stays test, val stays val, train stays train)
split_counts = {}
for rec in image_records:
    s = rec.get("split", "train")
    split_counts[s] = split_counts.get(s, 0) + 1
print("Split counts in manifest:", split_counts)

Dataset: kitchen hygiene gear 4
Classes (7): ['glove', 'hairnet', 'incorrect_mask', 'mask', 'no_glove', 'no_hairnet', 'no_mask']
Total image records: 31331
Split counts in manifest: {'test': 3028, 'train': 25175, 'val': 3128}


In [4]:
# ============================================================
# CELL 4: Download images and write YOLO-format label .txt files
# ============================================================
def download_and_label(rec):
    try:
        split = rec.get("split", "train")
        fname = rec["file"]
        img_path = IMAGES_DIR / split / fname
        label_path = LABELS_DIR / split / (Path(fname).stem + ".txt")

        # Download image if not already present
        if not img_path.exists():
            resp = requests.get(rec["url"], timeout=30)
            resp.raise_for_status()
            with open(img_path, "wb") as imgf:
                imgf.write(resp.content)

        # Write YOLO label file: class x_center y_center w h (already normalized)
        boxes = rec.get("annotations", {}).get("boxes", [])
        with open(label_path, "w") as lf:
            for b in boxes:
                cls, xc, yc, w, h = b
                lf.write(f"{int(cls)} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
        return True
    except Exception as e:
        print(f"Failed: {rec.get('file')} -> {e}")
        return False

MAX_WORKERS = 32
results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(download_and_label, rec) for rec in image_records]
    for f in tqdm(as_completed(futures), total=len(futures), desc="Downloading + labeling"):
        results.append(f.result())

print(f"Success: {sum(results)} / {len(results)}")

Failed: IMG_4513_JPG_jpg.rf.3a5afe37429e832a7658a63481ecd470.jpg -> HTTPSConnectionPool(host='cdn.ul.run', port=443): Max retries exceeded with url: /eu/i/4fe0b6c746937696625a0e98160d7f04.jpg?Expires=1785085162&KeyName=key-v1&Signature=i_4PKaDciHXo8DqWKqh-ymVcGpU (Caused by SSLError(SSLError(1, '[SSL: TLSV1_ALERT_DECODE_ERROR] tlsv1 alert decode error (_ssl.c:1010)')))


Success: 31330 / 31331


In [5]:
# ============================================================
# CELL 5: Write the dataset YAML for Ultralytics
# ============================================================
yaml_content = f"""
path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

nc: {NUM_CLASSES}
names: {CLASS_LIST}
"""

yaml_path = DATASET_ROOT / "data.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(yaml_content)

# Quick sanity check - do train/val folders actually have images?
for split in ["train", "val", "test"]:
    n_imgs = len(list((IMAGES_DIR / split).glob("*")))
    n_lbls = len(list((LABELS_DIR / split).glob("*")))
    print(f"{split}: {n_imgs} images, {n_lbls} label files")


path: /content/kitchen_hygiene_gear
train: images/train
val: images/val
test: images/test

nc: 7
names: ['glove', 'hairnet', 'incorrect_mask', 'mask', 'no_glove', 'no_hairnet', 'no_mask']

train: 25174 images, 25174 label files
val: 3128 images, 3128 label files
test: 3028 images, 3028 label files


In [10]:
# ============================================================
# CELL 6 (FAST): Train YOLOv11-nano, smaller image size, fewer epochs
# ============================================================
from ultralytics import YOLO

# model = YOLO("yolo11n.pt")   # nano instead of small — ~4x fewer params, much faster/epoch

# results = model.train(
#     data=str(yaml_path),
#     epochs=40,               # down from 100 — patience will stop earlier if it plateaus
#     imgsz=512,                # down from 640 — PPE items are still resolvable at 512
#     batch=-1,                 # auto-batch: picks the largest batch that fits your GPU's VRAM
#     patience=10,               # stop early if val mAP doesn't improve for 10 epochs
#     cache="ram",               # cache decoded images in RAM after first epoch (big speedup if it fits)
#     workers=8,                 # Colab allows more than the default despite 2 vCPUs shown
#     project="/content/runs",
#     name="hygiene_gear_yolo11n_fast",
#     optimizer="auto",
#     close_mosaic=10,           # turn off mosaic aug for the last 10 epochs (faster + stabilizes)
#     amp=True,
#     hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
#     translate=0.1,
#     scale=0.5,
#     fliplr=0.5,
#     device=0,
# )



# ============================================================
# CELL 6a (PROTOTYPE): Quick smoke-test run on a fraction of the data
# ============================================================
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(yaml_path),
    epochs=10,
    imgsz=416,
    batch=-1,
    fraction=0.25,      # trains on only 25% of the training images
    cache="ram",
    workers=8,
    project="/content/runs",
    name="hygiene_gear_prototype",
    device=0,
)

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/kitchen_hygiene_gear/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.25, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=hygiene_gear_prototype, nbs=64, nms=False, opset=None, optimize=False, opt

In [12]:
# ============================================================
# Find all trained weight files under /content/runs
# ============================================================
import glob

for p in glob.glob("/content/runs/**/weights/*.pt", recursive=True):
    print(p)

/content/runs/hygiene_gear_prototype/weights/best.pt
/content/runs/hygiene_gear_prototype/weights/last.pt


In [13]:
# ============================================================
# CELL 7: Validate on test split, report per-class metrics
# ============================================================
best_model_path = "/content/runs/hygiene_gear_prototype/weights/best.pt"
model = YOLO(best_model_path)

metrics = model.val(data=str(yaml_path), split="test")

print("\nPer-class mAP@0.5:")
for i, cls_name in enumerate(CLASS_LIST):
    print(f"  {cls_name:20s}: {metrics.box.ap50[i]:.4f}")

print(f"\nOverall mAP@0.5:      {metrics.box.map50:.4f}")
print(f"Overall mAP@0.5:0.95:  {metrics.box.map:.4f}")

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,517 parameters, 0 gradients, 6.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 21.5±7.0 MB/s, size: 68.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/kitchen_hygiene_gear/labels/test... 3028 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3028/3028 464.7it/s 6.5s
val: New cache created: /content/kitchen_hygiene_gear/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 190/190 8.2it/s 23.2s
                   all       3028       9563        0.5      0.424      0.397      0.185
                 glove        799       1956      0.452      0.348      0.336      0.162
               hairnet        653       1189      0.609      0.531      0.54

# 1st (Only 1 epoch) Result

In [9]:
# ============================================================
# CELL 8: Run inference on a sample image and visualize
# ============================================================
import matplotlib.pyplot as plt
from PIL import Image

sample_imgs = list((IMAGES_DIR / "test").glob("*.jpg"))[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, img_path in zip(axes.flatten(), sample_imgs):
    result = model.predict(source=str(img_path), conf=0.35, verbose=False)[0]
    annotated = result.plot()  # BGR numpy array with boxes drawn
    ax.imshow(annotated[..., ::-1])  # BGR -> RGB
    ax.axis("off")
    ax.set_title(img_path.name, fontsize=9)

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

# 10 epoch Result


In [14]:
# ============================================================
# CELL 8: Run inference on a sample image and visualize
# ============================================================
import matplotlib.pyplot as plt
from PIL import Image

sample_imgs = list((IMAGES_DIR / "test").glob("*.jpg"))[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, img_path in zip(axes.flatten(), sample_imgs):
    result = model.predict(source=str(img_path), conf=0.35, verbose=False)[0]
    annotated = result.plot()  # BGR numpy array with boxes drawn
    ax.imshow(annotated[..., ::-1])  # BGR -> RGB
    ax.axis("off")
    ax.set_title(img_path.name, fontsize=9)

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [16]:
# ============================================================
# CELL: Download best.pt straight to your local machine
# ============================================================
from google.colab import files

best_pt_path = f"/content/runs/hygiene_gear_prototype/weights/best.pt"
files.download(best_pt_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>